**GRADIO APP FOR GETTING GEODATA:**

Welcome! Welcome to our humble app!
Here you will be able to get accurate goegraphical information with just a quick prompt. Our app uses Chatgpt to interact with the users. We have made two functions that allow chat to get geo data and incorporate it into the response.

# **installs and imports**

In [ ]:
!pip install openai
!pip install PyMuPDF
!pip install gradio

# osm
!pip install osmnx
!pip install geopy
!pip install scikit-learn

# gee topo
!pip install
!pip install earthengine-api
!pip install folium
!pip install geemap
!pip install geopandas
!pip install rasterio
!pip install numpy pillow

In [ ]:
import gradio as gr
import openai
from openai import OpenAI

import geopandas as gpd
import folium
import numpy as np
import os
import ee
import geemap
import rasterio
import osmnx as ox
import matplotlib.pyplot as plt
from PIL import Image
from geopy.geocoders import Nominatim
from google.colab import files

# **PLEASE INSERT YOUR OWN CREDENTIALS!!!!!!!**

In [ ]:
client = OpenAI(api_key="insert_yours")

In [ ]:
# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='ee-insert yours')

# **GEO FUNCTIONS TO USE**

In [ ]:
# make image directory
os.makedirs("/content/img", exist_ok=True)

# Function to get city boundaries from OpenStreetMap (OSM)
def get_osm_boundary(city_name):
    geolocator = Nominatim(user_agent="geoapi")
    location = geolocator.geocode(city_name, exactly_one=True)

    if location:
        bbox = location.raw['boundingbox']
        lat_min, lat_max, lon_min, lon_max = map(float, bbox)
        return ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])  # Use OSM boundary
    else:
        print(f"❌ Error: Unable to find city '{city_name}' in OSM.")
        return None

def define_file_paths(city_name):
    base_path = "/content/img"
    return {
        "elevation_tiff": os.path.join(base_path, f"{city_name}_elevation.tif"),
        "elevation_png": os.path.join(base_path, f"{city_name}_elevation.png"),
        "street_network": os.path.join(base_path, f"{city_name}_street_network.png")
    }

# Function to generate geographic data (Street Layout or Topography)
def generateGeoData(operation_type, city_name):

    if operation_type == "layout":
        try:
            graph = ox.graph_from_place(city_name, network_type='drive')
            fig, ax = ox.plot_graph(graph, node_size=0, bgcolor='#FFFFFF', edge_color='#000000', edge_linewidth=0.2)

            image_path = define_file_paths(city_name)["street_network"]
            fig.savefig(image_path, dpi=400, bbox_inches='tight')
            print(f"✅ Street layout saved: {image_path}")
            return image_path

        except Exception as e:
            print(f"❌ Error generating street layout: {e}")
            return None

    elif operation_type == "topography":
        print(f"🔍 Searching for {city_name} in OpenStreetMap...")

        # Fetch city boundary from OSM
        bbox = get_osm_boundary(city_name)
        if bbox is None:
            print(f"❌ Error: Could not find {city_name} boundary in OSM.")
            return None

        print(f"✅ City boundary found! Clipping elevation map to {city_name}...")

        dataset = ee.Image('NASA/NASADEM_HGT/001')
        elevation = dataset.select('elevation')

        elevation_city = elevation.clip(bbox)

        paths = define_file_paths(city_name)
        tif_file, png_file = paths["elevation_tiff"], paths["elevation_png"]

        # Export the elevation map as GeoTIFF
        export_scale = 100 #HERE TO CHANGE THE SCALE
        geemap.ee_export_image(
            elevation_city, filename=tif_file, scale=export_scale, region=bbox, file_per_band=False
        )
        print(f"✅ GeoTIFF saved: {tif_file}")

        ##### Convert GeoTIFF to PNG !!!!!
        if os.path.exists(tif_file):
            with rasterio.open(tif_file) as src:
                array = src.read(1)  # Read first band
                array[array == src.nodata] = 0  # Replace NoData values with 0

                # Normalize elevation values (0-255 grayscale)
                min_val, max_val = src.read(1).min(), src.read(1).max()
                normalized_array = ((array - min_val) / (max_val - min_val)) * 255
                normalized_array = normalized_array.astype("uint8")

                # Convert to grayscale image
                img = Image.fromarray(normalized_array)
                img = img.convert("L")

                # Save the PNG
                img.save(png_file)

                # Show the processed elevation map
                plt.figure(figsize=(8, 8))
                plt.imshow(img, cmap="gray")
                plt.axis("off")
                plt.title(f"{city_name} Elevation Map")
                plt.show()

            print(f"✅ PNG saved: {png_file}")

            files.download(png_file)

            return png_file

        else:
            print("❌ Error: GeoTIFF file not found.")
            return None

    else:
        return "❌ Error: Invalid operation type."

# **PROMPT TEMPLATES AND GRADIO APP LAUNCHER**

In [ ]:
prompt_template = """
You are a helpful assistant. Try your best to fulfill the user query. To empower
you further, you can use the "generateGeoData" tool. In order to use it, answer in the following
JSON format:

{
  "message": "your regular message to the user",
  "tool": "generateGeoData",
  "args": {"operation_type": "layout", "city_name": "Barcelona"}
}

If you don't need to use the tool or you have already used it and retrieved its answer, leave "tool" and "args" empty. If you use the tool,
use the message field to explain your argument choices.
The argument "operation_type" can either be "layout" (to get the map of the streets of a city) or
"topography" (to get the topographical features of the city).

Ensure to always answer in exactly that JSON format. Do not add any comments or thoughts.
Below is the user query:
"""


In [ ]:
user_prompt = "show me the topography of Buenos Aires"

In [ ]:
def callGPT_JSONMode(user_prompt, model="gpt-4o"):
    conversationLog = [
        {"role": "system", "content": prompt_template},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=conversationLog,
    )

    return response.choices[0].message.model_dump()["content"]

# Gradio interface function (with GPT-4o integration)
def generate_city_map_with_GPT(user_prompt):
    # Call GPT-4o to analyze the prompt
    gpt_response = callGPT_JSONMode(user_prompt)

    # Parse GPT response
    try:
        response_json = eval(gpt_response)
        message = response_json.get("message", "❌ GPT did not return a valid message.")
        tool = response_json.get("tool", "")
        args = response_json.get("args", {})

        if tool == "generateGeoData" and args:
            operation_type = args.get("operation_type", "unknown")
            city_name = args.get("city_name", "unknown")

            if operation_type not in ["layout", "topography"]:
                return f"❌ Error: Invalid operation type detected: {operation_type}", None, None

            # Call generateGeoData using the detected parameters
            output_file = generateGeoData(operation_type, city_name)

            if output_file is None:
                return f"❌ Unable to generate {operation_type} map for {city_name}.", None, None

            return message, output_file, output_file
        else:
            return message, None, None

    except Exception as e:
        return f"❌ Error parsing GPT response: {e}", None, None

interface = gr.Interface(
    fn=generate_city_map_with_GPT,
    inputs=gr.Textbox(label="Enter a prompt (e.g., 'Generate a street map of Barcelona', 'Show me Berlin topography')"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Image(label="Preview", type="filepath"),
        gr.File(label="Download Map")
    ],
    title="AI-Powered City Map Generator",
    description="Enter a city name and the type of map you want. AI will analyze your prompt and generate a city map accordingly."
)

interface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57c296834a21ec979e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
